# Retrieval Evaluation Dataset Generator

Generates ~100 ground-truth QA pairs from Pali Canon chunks using Groq (llama-3.3-70b-versatile).

**Pipeline:**
1. Sample ~200 chunks from database (stratified by document)
2. Generate question + answer per chunk via Groq
3. Filter with 3 critique agents (groundedness, standalone, relevance)
4. Deduplicate via embedding similarity
5. Export to `data/eval_dataset.jsonl`

In [ ]:
import asyncio
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

# Validate required env vars
required_vars = ["GROQ_API_KEY", "DB_URL"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing env vars: {missing}")

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print("✓ Groq client initialized")
print(f"✓ DB_URL set: {os.getenv('DB_URL')[:30]}...")

In [ ]:
import sys
sys.path.insert(0, "..")

from db.database import session_scope
from db.schema import Chunk, Document

with session_scope() as session:
    # Load all chunks with parent document title
    results = (
        session.query(
            Chunk.id,
            Chunk.uuid,
            Chunk.chunk_text,
            Chunk.chunk_index,
            Chunk.chunk_metadata,
            Chunk.document_id,
            Document.title,
        )
        .join(Document, Chunk.document_id == Document.id)
        .all()
    )

all_chunks = [
    {
        "id": r.id,
        "uuid": r.uuid,
        "chunk_text": r.chunk_text,
        "chunk_index": r.chunk_index,
        "chunk_metadata": r.chunk_metadata,
        "document_id": r.document_id,
        "document_title": r.title,
    }
    for r in results
]

print(f"✓ Loaded {len(all_chunks)} chunks from {len(set(c['document_id'] for c in all_chunks))} documents")

# Show distribution
doc_counts = {}
for c in all_chunks:
    doc_counts[c["document_title"]] = doc_counts.get(c["document_title"], 0) + 1
for title, count in sorted(doc_counts.items()):
    print(f"  {title}: {count} chunks")

In [ ]:
SAMPLE_SIZE = 200
random.seed(42)

# Group chunks by document
from collections import defaultdict
chunks_by_doc = defaultdict(list)
for chunk in all_chunks:
    chunks_by_doc[chunk["document_id"]].append(chunk)

# Proportional sampling per document
total_chunks = len(all_chunks)
sampled_chunks = []
for doc_id, doc_chunks in chunks_by_doc.items():
    proportion = len(doc_chunks) / total_chunks
    n_samples = max(1, round(SAMPLE_SIZE * proportion))
    sampled = random.sample(doc_chunks, min(n_samples, len(doc_chunks)))
    sampled_chunks.extend(sampled)

# Trim to exact target if oversampled
if len(sampled_chunks) > SAMPLE_SIZE:
    sampled_chunks = random.sample(sampled_chunks, SAMPLE_SIZE)

print(f"✓ Sampled {len(sampled_chunks)} chunks (target: {SAMPLE_SIZE})")

# Verify distribution
sampled_doc_counts = {}
for c in sampled_chunks:
    sampled_doc_counts[c["document_title"]] = sampled_doc_counts.get(c["document_title"], 0) + 1
for title, count in sorted(sampled_doc_counts.items()):
    print(f"  {title}: {count} sampled chunks")

In [ ]:
QA_GENERATION_PROMPT = """Your task is to write a question and answer given a passage from Buddhist scripture (Pali Canon).

Requirements:
- The question should be answerable from the passage
- Phrase it as a real meditation practitioner or student might ask
- Do NOT reference "the passage", "the text", or "according to" — write as if asking a teacher
- The answer should be concise (1-3 sentences) and grounded strictly in the passage
- Classify the question type as one of: factual, conceptual, practical, cross_textual, pali_specific

Definitions:
- factual: Answerable with a specific fact from the passage ("What are the four...?")
- conceptual: Requires understanding relationships or meaning ("How does X relate to Y?")
- practical: About meditation practice or technique ("How does one develop...?")
- cross_textual: Could relate to themes across multiple texts ("What role does X play in...?")
- pali_specific: Uses or asks about Pali terminology ("What does X mean?")

Context: {chunk_text}

Reply with ONLY valid JSON (no markdown, no code fences):
{{"question": "...", "answer": "...", "question_type": "..."}}"""


def generate_qa(chunk: dict, max_retries: int = 2) -> dict | None:
    """Generate a QA pair from a chunk using Groq."""
    for attempt in range(max_retries + 1):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "user", "content": QA_GENERATION_PROMPT.format(chunk_text=chunk["chunk_text"])}
                ],
                temperature=0.7,
                max_tokens=500,
            )
            content = response.choices[0].message.content.strip()
            # Handle potential markdown code fences
            if content.startswith("```"):
                content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            parsed = json.loads(content)

            # Validate required fields
            if not all(k in parsed for k in ("question", "answer", "question_type")):
                raise ValueError("Missing required fields")

            return {
                "question": parsed["question"],
                "answer": parsed["answer"],
                "question_type": parsed["question_type"],
                "chunk_uuid": chunk["uuid"],
                "chunk_text": chunk["chunk_text"],
                "document_id": chunk["document_id"],
                "document_title": chunk["document_title"],
            }
        except (json.JSONDecodeError, ValueError, KeyError) as e:
            if attempt < max_retries:
                time.sleep(1)
                continue
            print(f"  ✗ Failed for chunk {chunk['uuid'][:8]}: {e}")
            return None
        except Exception as e:
            print(f"  ✗ API error for chunk {chunk['uuid'][:8]}: {e}")
            time.sleep(2)
            if attempt < max_retries:
                continue
            return None

print("✓ QA generation function defined")

In [ ]:
# Groq free tier: 30 requests/min
RATE_LIMIT_DELAY = 2.1  # seconds between requests (safe margin)

raw_qa_pairs = []
failed_count = 0

print(f"Generating QA pairs for {len(sampled_chunks)} chunks...")
print(f"Estimated time: ~{len(sampled_chunks) * RATE_LIMIT_DELAY / 60:.1f} minutes")

for i, chunk in enumerate(sampled_chunks):
    if i > 0 and i % 10 == 0:
        print(f"  Progress: {i}/{len(sampled_chunks)} ({len(raw_qa_pairs)} generated, {failed_count} failed)")

    result = generate_qa(chunk)
    if result:
        raw_qa_pairs.append(result)
    else:
        failed_count += 1

    time.sleep(RATE_LIMIT_DELAY)

print(f"\n✓ Generated {len(raw_qa_pairs)} QA pairs ({failed_count} failures)")
print(f"Success rate: {len(raw_qa_pairs) / len(sampled_chunks) * 100:.1f}%")

# Preview question type distribution
type_counts = {}
for qa in raw_qa_pairs:
    t = qa["question_type"]
    type_counts[t] = type_counts.get(t, 0) + 1
print(f"\nQuestion type distribution:")
for t, count in sorted(type_counts.items()):
    print(f"  {t}: {count}")

In [ ]:
GROUNDEDNESS_PROMPT = """You are evaluating a question-answer pair generated from a passage.

Rate the GROUNDEDNESS of the question on a scale of 1-5:
1 = Question cannot be answered from this passage at all
2 = Question is only loosely related to the passage
3 = Question is partially answerable from the passage
4 = Question is mostly answerable from the passage
5 = Question is fully and directly answerable from the passage

Passage: {chunk_text}
Question: {question}
Answer: {answer}

Reply with ONLY valid JSON: {{"score": <int>, "reason": "<one sentence>"}}"""

STANDALONE_PROMPT = """You are evaluating whether a question makes sense on its own, without needing to see the source passage.

Rate the STANDALONE quality on a scale of 1-5:
1 = Impossible to understand without context (e.g., "What does the above describe?")
2 = Very unclear without context
3 = Somewhat understandable but vague
4 = Mostly clear and self-contained
5 = Perfectly clear question that anyone could understand

Question: {question}

Reply with ONLY valid JSON: {{"score": <int>, "reason": "<one sentence>"}}"""

RELEVANCE_PROMPT = """You are evaluating whether a question about Buddhist meditation and philosophy is realistic.

Rate the RELEVANCE on a scale of 1-5:
1 = No real practitioner would ask this (too artificial or academic)
2 = Very unlikely question
3 = Possible but unusual
4 = A practitioner might reasonably ask this
5 = Very natural question a meditator or student would ask

Question: {question}

Reply with ONLY valid JSON: {{"score": <int>, "reason": "<one sentence>"}}"""


def score_qa(qa: dict, prompt_template: str, prompt_kwargs: dict, max_retries: int = 2) -> int | None:
    """Score a QA pair using a Groq critique agent. Returns score 1-5 or None on failure."""
    for attempt in range(max_retries + 1):
        try:
            response = groq_client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "user", "content": prompt_template.format(**prompt_kwargs)}
                ],
                temperature=0.0,
                max_tokens=100,
            )
            content = response.choices[0].message.content.strip()
            if content.startswith("```"):
                content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            parsed = json.loads(content)
            score = int(parsed["score"])
            if 1 <= score <= 5:
                return score
            return None
        except Exception:
            if attempt < max_retries:
                time.sleep(2)
                continue
            return None

print("✓ Critique agent functions defined")

In [ ]:
FILTER_THRESHOLD = 4

print(f"Filtering {len(raw_qa_pairs)} QA pairs with 3 critique agents...")
print(f"Total API calls: ~{len(raw_qa_pairs) * 3}")
print(f"Estimated time: ~{len(raw_qa_pairs) * 3 * RATE_LIMIT_DELAY / 60:.1f} minutes")

scored_qa_pairs = []

for i, qa in enumerate(raw_qa_pairs):
    if i > 0 and i % 20 == 0:
        passed_so_far = sum(1 for s in scored_qa_pairs if s.get("passed", False))
        print(f"  Progress: {i}/{len(raw_qa_pairs)} ({passed_so_far} passed so far)")

    # Score groundedness
    g_score = score_qa(qa, GROUNDEDNESS_PROMPT, {
        "chunk_text": qa["chunk_text"],
        "question": qa["question"],
        "answer": qa["answer"],
    })
    time.sleep(RATE_LIMIT_DELAY)

    # Score standalone quality
    s_score = score_qa(qa, STANDALONE_PROMPT, {
        "question": qa["question"],
    })
    time.sleep(RATE_LIMIT_DELAY)

    # Score relevance
    r_score = score_qa(qa, RELEVANCE_PROMPT, {
        "question": qa["question"],
    })
    time.sleep(RATE_LIMIT_DELAY)

    qa_scored = {
        **qa,
        "groundedness_score": g_score,
        "standalone_score": s_score,
        "relevance_score": r_score,
        "passed": all(
            s is not None and s >= FILTER_THRESHOLD
            for s in [g_score, s_score, r_score]
        ),
    }
    scored_qa_pairs.append(qa_scored)

passed = [qa for qa in scored_qa_pairs if qa["passed"]]
failed = [qa for qa in scored_qa_pairs if not qa["passed"]]

print(f"\n✓ Filtering complete")
print(f"  Passed: {len(passed)} ({len(passed)/len(scored_qa_pairs)*100:.1f}%)")
print(f"  Failed: {len(failed)} ({len(failed)/len(scored_qa_pairs)*100:.1f}%)")

# Show score distributions
for metric in ["groundedness_score", "standalone_score", "relevance_score"]:
    scores = [qa[metric] for qa in scored_qa_pairs if qa[metric] is not None]
    print(f"  {metric}: avg={sum(scores)/len(scores):.2f}, min={min(scores)}, max={max(scores)}")

In [ ]:
from langchain_voyageai import VoyageAIEmbeddings
import numpy as np

# Embed all passing questions
voyage = VoyageAIEmbeddings(model="voyage-3.5")
questions = [qa["question"] for qa in passed]
question_embeddings = voyage.embed_documents(questions)

print(f"✓ Embedded {len(question_embeddings)} questions")

# Find near-duplicates (cosine similarity > 0.9)
embeddings_array = np.array(question_embeddings)
# Normalize for cosine similarity
norms = np.linalg.norm(embeddings_array, axis=1, keepdims=True)
normalized = embeddings_array / norms

similarity_matrix = normalized @ normalized.T

# Find pairs to remove (keep the first, remove the second)
to_remove = set()
for i in range(len(passed)):
    if i in to_remove:
        continue
    for j in range(i + 1, len(passed)):
        if j in to_remove:
            continue
        if similarity_matrix[i][j] > 0.9:
            to_remove.add(j)

deduplicated = [qa for i, qa in enumerate(passed) if i not in to_remove]

print(f"✓ Removed {len(to_remove)} near-duplicate questions")
print(f"✓ Final dataset: {len(deduplicated)} QA pairs")

In [ ]:
OUTPUT_PATH = Path("../data/eval_dataset.jsonl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

records = []
for i, qa in enumerate(deduplicated):
    record = {
        "id": f"eval_{i+1:03d}",
        "question": qa["question"],
        "reference_answer": qa["answer"],
        "reference_contexts": [qa["chunk_text"]],
        "chunk_ids": [qa["chunk_uuid"]],
        "source_document_id": qa["document_id"],
        "metadata": {
            "question_type": qa["question_type"],
            "groundedness_score": qa["groundedness_score"],
            "standalone_score": qa["standalone_score"],
            "relevance_score": qa["relevance_score"],
            "source_document_title": qa["document_title"],
        },
    }
    records.append(record)

with open(OUTPUT_PATH, "w") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")

print(f"✓ Saved {len(records)} records to {OUTPUT_PATH}")

In [ ]:
df = pd.DataFrame([
    {
        "id": r["id"],
        "question_type": r["metadata"]["question_type"],
        "groundedness": r["metadata"]["groundedness_score"],
        "standalone": r["metadata"]["standalone_score"],
        "relevance": r["metadata"]["relevance_score"],
        "source_doc": r["metadata"]["source_document_title"],
        "question_length": len(r["question"]),
        "answer_length": len(r["reference_answer"]),
    }
    for r in records
])

print("=== Eval Dataset Summary ===")
print(f"Total QA pairs: {len(df)}")
print(f"\nBy question type:")
print(df["question_type"].value_counts().to_string())
print(f"\nBy source document:")
print(df["source_doc"].value_counts().to_string())
print(f"\nScore distributions:")
for col in ["groundedness", "standalone", "relevance"]:
    print(f"  {col}: mean={df[col].mean():.2f}, std={df[col].std():.2f}")
print(f"\nQuestion length: mean={df['question_length'].mean():.0f}, std={df['question_length'].std():.0f}")
print(f"Answer length: mean={df['answer_length'].mean():.0f}, std={df['answer_length'].std():.0f}")